# Лекция 1. От текста к проверяемому циклу

Все ответы и данные в этом ноутбуке заданы учебным кодом; LLM не вызывается. Для запуска достаточно Python 3.10+ и среды Jupyter Notebook. Ключи, интернет, GPU и установка библиотек не нужны.

Достаточно понимать функцию, словарь, список и цикл Python. На четырёх примерах вы научитесь отличать текст от действия, читать трассу `policy(history)`, находить проверки аргументов и бюджета, объяснять, почему выигрыш на dev может исчезнуть на test.

Запускайте ячейки сверху вниз. Здесь приведены готовые примеры; задания для самостоятельного решения находятся в `01-agent-student.ipynb`.


## 1. Написать «сохранено» — ещё не сохранить

**Интуиция.** Ответ модели является данными. Изменение среды происходит, когда окружающая программа действительно вызывает функцию записи.

**Формально.** Генерация текста даёт сообщение $y$. Сама по себе она не задаёт переход среды $s\to s'$. Действие исполняется отдельной функцией: $s'=T(s,a)$.

**Пример.** Среда ниже — обычный словарь в памяти, «заметка» — строка. Это не запись файла и не отправка сообщения. Ответ, приписанный модели, также задан вручную.

**Перед запуском:** сколько заметок появится после одного `print(scripted_message)`?

In [1]:
from copy import deepcopy
import json
from typing import NamedTuple

local_world = {"notes": []}
scripted_message = "Заметка сохранена."  # Заранее написанный текст; вызова LLM нет.
initial_world = deepcopy(local_world)
print("SCRIPTED сообщение:", scripted_message)
assert local_world == initial_world
print("После сообщения:", local_world)

def save_note(world, text):
    world["notes"].append(text)
    return {"ok": True, "index": len(world["notes"]) - 1}

receipt = save_note(local_world, "Проверить аудиторию по расписанию")
assert receipt["ok"] and local_world["notes"][receipt["index"]] == "Проверить аудиторию по расписанию"
print("После вызова функции:", local_world)
print("Свидетельство:", receipt)

SCRIPTED сообщение: Заметка сохранена.
После сообщения: {'notes': []}
После вызова функции: {'notes': ['Проверить аудиторию по расписанию']}
Свидетельство: {'ok': True, 'index': 0}


**Самопроверка.** До вызова `save_note` заметок **0**. После вызова — **1**; мы проверили и результат функции, и состояние словаря. Одной фразы об успехе было бы недостаточно.

## 2. SCRIPTED-политика читает историю

**Интуиция.** Следующее действие выбирается после наблюдения. Нужно узнать два поля: аудиторию и источник. Пока поля нет в истории, политика предлагает `lookup`; когда оба получены — `finish`.

**Формально.** $h_t=(x,a_0,o_1,\ldots,a_{t-1},o_t)$, $a_t=\pi_{\mathrm{scripted}}(h_t)$. Здесь $\pi$ — написанная нами детерминированная функция, **не языковая модель**. Она показывает контракт цикла и зависимость от истории; её прохождение не доказывает способности LLM выбирать действия.

Среда вымышлена. В настоящей задаче нужно также проверить курс, дату и актуальность документа; здесь мы изолируем только механизм чтения двух полей.

In [2]:
SCHEDULE = {
    "room": "314",
    "source": "учебное-расписание-v1 (вымышленный источник)",
}
GOAL = "Узнай аудиторию следующего занятия и укажи источник."

def observed_fields(history):
    return {
        message["key"]: message["value"]
        for message in history
        if message.get("role") == "tool" and message.get("name") == "lookup"
    }

def policy(history):
    """SCRIPTED: правило по истории. Ни генерации текста, ни вызова LLM."""
    known = observed_fields(history)
    for key in ("room", "source"):
        if key not in known:
            return {"tool": "lookup", "args": {"key": key}}
    return {"tool": "finish", "args": {"answer": known}}

print("Пустая история наблюдений →", policy([{"role": "user", "content": GOAL}]))

Пустая история наблюдений → {'tool': 'lookup', 'args': {'key': 'room'}}


### Сначала контракт, затем исполнение

**Интуиция.** Структурированный ответ — предложение действия. Harness проверяет его до обращения к среде.

В примере допустимы ровно два формата:

```python
{"tool": "lookup", "args": {"key": "room"}}  # key: room или source
{"tool": "finish", "args": {"answer": {"room": "...", "source": "..."}}}
```

Проверяем точный набор полей, имена инструментов и типы аргументов. Произвольного `eval` или исполнения кода из ответа здесь нет. Политика возвращает Python-словарь. При подключении LLM адаптеру также потребуется разбирать JSON-строку.

In [3]:
def validate_action(action):
    if type(action) is not dict or set(action) != {"tool", "args"}:
        raise ValueError("Ожидается объект с полями tool и args")
    tool, args = action["tool"], action["args"]
    if type(tool) is not str or tool not in {"lookup", "finish"}:
        raise ValueError("Инструмент не разрешён")
    if type(args) is not dict:
        raise ValueError("args должен быть словарём")
    if tool == "lookup":
        if set(args) != {"key"} or type(args["key"]) is not str:
            raise ValueError("lookup принимает единственный строковый аргумент key")
        if args["key"] not in {"room", "source"}:
            raise ValueError("lookup разрешён только для room и source")
    else:
        answer = args.get("answer")
        if set(args) != {"answer"} or type(answer) is not dict:
            raise ValueError("finish принимает словарь answer")
        if set(answer) != {"room", "source"}:
            raise ValueError("В answer нужны ровно room и source")
        if any(type(value) is not str or not value.strip() for value in answer.values()):
            raise ValueError("Поля answer должны быть непустыми строками")
    return tool, deepcopy(args)

print("Допустимое действие:", validate_action(policy([])))

Допустимое действие: ('lookup', {'key': 'room'})


### Где живёт цикл

Harness ниже хранит историю, проверяет действие, вызывает чтение среды и возвращает наблюдение политике. Чтение словаря выполняется внутри `run`.

**Бюджет $B=3$ означает три решения политики.** Ошибочное действие и `finish` тоже расходуют по одному шагу. Поэтому нормальный путь — `lookup(room) → lookup(source) → finish` — укладывается ровно в бюджет. При исчерпании бюджета нового вызова политики не будет.

Отдельная проверка результата требует оба наблюдения и совпадение ответа с ними и средой. `status="finished"` сообщает об остановке по `finish`; только `verified=True` означает выполнение **нашего узкого учебного критерия**. Эти поля отвечают на разные вопросы.

In [4]:
def run(decide, environment, max_steps=3):
    if type(max_steps) is not int or max_steps <= 0:
        raise ValueError("max_steps должен быть положительным целым")
    history = [{"role": "user", "content": GOAL}]
    trace, tool_calls = [], 0
    for step in range(1, max_steps + 1):
        action = decide(deepcopy(history))
        history.append({"role": "assistant", "action": deepcopy(action)})
        try:
            tool, args = validate_action(action)
        except ValueError as error:
            observation = {"role": "tool", "name": "validation", "error": str(error)}
            history.append(observation)
            trace.append({"step": step, "action": action, "observation": observation})
            continue  # Отказ тоже потребляет шаг; инструмент не запущен.
        if tool == "finish":
            answer = args["answer"]
            known = observed_fields(history)
            verified = all(
                key in known and answer[key] == known[key] == environment[key]
                for key in ("room", "source")
            )
            trace.append({"step": step, "action": action, "observation": {"verified": verified}})
            return {"status": "finished", "verified": verified, "answer": answer,
                    "steps": step, "tool_calls": tool_calls, "history": history, "trace": trace}
        key = args["key"]
        tool_calls += 1
        observation = {"role": "tool", "name": "lookup", "key": key, "value": environment[key]}
        history.append(observation)
        trace.append({"step": step, "action": action, "observation": observation})
    return {"status": "budget_exhausted", "verified": False, "answer": None,
            "steps": max_steps, "tool_calls": tool_calls, "history": history, "trace": trace}

def show_trace(result):
    for event in result["trace"]:
        print(json.dumps(event, ensure_ascii=False))
    print("Итог:", {key: result[key] for key in ("status", "verified", "steps", "tool_calls")})

In [5]:
scripted_run = run(policy, SCHEDULE, max_steps=3)
print("SCRIPTED TRACE — вызовов LLM: 0")
show_trace(scripted_run)
assert scripted_run["verified"] and scripted_run["steps"] == 3
assert scripted_run["tool_calls"] == 2
assert scripted_run["answer"] == SCHEDULE

# Новое наблюдение меняет решение, хотя код политики остаётся прежним.
only_room = [{"role": "tool", "name": "lookup", "key": "room", "value": "314"}]
assert policy(only_room) == {"tool": "lookup", "args": {"key": "source"}}
print("Если room уже наблюдалась, следующее действие:", policy(only_room))

SCRIPTED TRACE — вызовов LLM: 0
{"step": 1, "action": {"tool": "lookup", "args": {"key": "room"}}, "observation": {"role": "tool", "name": "lookup", "key": "room", "value": "314"}}
{"step": 2, "action": {"tool": "lookup", "args": {"key": "source"}}, "observation": {"role": "tool", "name": "lookup", "key": "source", "value": "учебное-расписание-v1 (вымышленный источник)"}}
{"step": 3, "action": {"tool": "finish", "args": {"answer": {"room": "314", "source": "учебное-расписание-v1 (вымышленный источник)"}}}, "observation": {"verified": true}}
Итог: {'status': 'finished', 'verified': True, 'steps': 3, 'tool_calls': 2}
Если room уже наблюдалась, следующее действие: {'tool': 'lookup', 'args': {'key': 'source'}}


**Самопроверка.** Кто прочитал расписание? `run`, а не текст политики. Почему второе действие отличается от первого? В истории появилось наблюдение `room`. Почему завершение проверено? Получены **оба** поля и ответ совпадает с наблюдениями и средой.

## 3. Неверное действие и конечный бюджет

**Интуиция.** Запреты должны действовать даже тогда, когда политика повторяет одну ошибку. Просьба «не зацикливайся» не заменяет счётчик.

**Формально.** До исполнения требуется $a_t\in\mathcal A_{\mathrm{allowed}}$. Любое решение уменьшает остаток бюджета: $b_{t+1}=b_t-1$; при $b_t=0$ цикл останавливается. Валидность действия не означает, что оно полезно для задачи.

**Перед запуском:** что произойдёт, если вместо строкового ключа передать список? Сколько раз будет вызван инструмент?

In [6]:
def invalid_policy(history):
    return {"tool": "lookup", "args": {"key": ["room"]}}

rejected = run(invalid_policy, SCHEDULE, max_steps=3)
show_trace(rejected)
assert rejected["status"] == "budget_exhausted"
assert rejected["steps"] == 3 and rejected["tool_calls"] == 0
assert all("error" in event["observation"] for event in rejected["trace"])

# Неразрешённое имя действия также отклоняется до исполнения.
try:
    validate_action({"tool": "send_message", "args": {"text": "Здравствуйте"}})
except ValueError as error:
    print("Проверка полномочий:", error)
else:
    raise AssertionError("Неразрешённое действие было принято")

{"step": 1, "action": {"tool": "lookup", "args": {"key": ["room"]}}, "observation": {"role": "tool", "name": "validation", "error": "lookup принимает единственный строковый аргумент key"}}
{"step": 2, "action": {"tool": "lookup", "args": {"key": ["room"]}}, "observation": {"role": "tool", "name": "validation", "error": "lookup принимает единственный строковый аргумент key"}}
{"step": 3, "action": {"tool": "lookup", "args": {"key": ["room"]}}, "observation": {"role": "tool", "name": "validation", "error": "lookup принимает единственный строковый аргумент key"}}
Итог: {'status': 'budget_exhausted', 'verified': False, 'steps': 3, 'tool_calls': 0}
Проверка полномочий: Инструмент не разрешён


**Ответ:** три отказа, **ноль** обращений к среде, затем `budget_exhausted`. Ошибка возвращается как наблюдение; наша намеренно плохая SCRIPTED-политика его игнорирует.

Теперь два других случая: политика бесконечно повторяет **допустимое** чтение; политика сразу запрашивает `finish` с правдоподобным ответом, ничего не прочитав.

**Предскажите:** какое условие остановит первый запуск? Будет ли второй считаться проверенным успехом? Здесь полезно различать «остановился» и «решил задачу».

In [7]:
def repeating_policy(history):
    return {"tool": "lookup", "args": {"key": "room"}}

repeated = run(repeating_policy, SCHEDULE, max_steps=3)
print("Повтор допустимого чтения:")
show_trace(repeated)
assert repeated["status"] == "budget_exhausted"
assert repeated["steps"] == 3 and repeated["tool_calls"] == 3
assert repeated["answer"] is None and not repeated["verified"]

def premature_finish(history):
    return {"tool": "finish", "args": {"answer": dict(SCHEDULE)}}

premature = run(premature_finish, SCHEDULE, max_steps=3)
print("Завершение без наблюдений:")
show_trace(premature)
assert premature["status"] == "finished" and not premature["verified"]
assert premature["tool_calls"] == 0

Повтор допустимого чтения:
{"step": 1, "action": {"tool": "lookup", "args": {"key": "room"}}, "observation": {"role": "tool", "name": "lookup", "key": "room", "value": "314"}}
{"step": 2, "action": {"tool": "lookup", "args": {"key": "room"}}, "observation": {"role": "tool", "name": "lookup", "key": "room", "value": "314"}}
{"step": 3, "action": {"tool": "lookup", "args": {"key": "room"}}, "observation": {"role": "tool", "name": "lookup", "key": "room", "value": "314"}}
Итог: {'status': 'budget_exhausted', 'verified': False, 'steps': 3, 'tool_calls': 3}
Завершение без наблюдений:
{"step": 1, "action": {"tool": "finish", "args": {"answer": {"room": "314", "source": "учебное-расписание-v1 (вымышленный источник)"}}}, "observation": {"verified": false}}
Итог: {'status': 'finished', 'verified': False, 'steps': 1, 'tool_calls': 0}


**Самопроверка.** Повтор остановил **harness по бюджету**. Второй запуск завершён, но проверку не прошёл: наблюдений нет. Даже случайно угаданный правильный ответ не удовлетворяет нашему заранее заданному требованию получить свидетельства.

## 4. Синтетический авторесерч: на dev лучше, на test хуже

**Интуиция.** Автоматизация выбора усиливает и полезный сигнал, и подгонку под знакомые примеры. Проверим три заранее написанные стратегии, выберем одну на dev и только затем раскроем test.

**Это учебная симуляция.** Три стратегии заданы в коде; LLM не предлагает изменения, веса не обучаются. Мы демонстрируем только этапы «кандидат → измерение → выбор → независимая проверка». Данные намеренно сконструированы так, чтобы показать переобучение; числа ниже нельзя переносить на реальные агенты.

**Формально.** Для бинарных меток используем accuracy:

$$\widehat S_D(f)=\frac1{|D|}\sum_{(x,y)\in D}\mathbf1[f(x)=y],\qquad
f^*=\arg\max_{f\in\mathcal F}\widehat S_{\mathrm{dev}}(f).$$

В синтетическом мире правило — $y=\mathbf1[x\ge6]$. На dev изменены две метки (при $x=2$ и $x=7$). Это заданный шум, а не случайный запуск. Test содержит другие идентификаторы и значения $x$, без этих искажений.

In [8]:
class Example(NamedTuple):
    uid: str
    x: float
    y: int

# Фиксированный dev: 10 строк, включая две намеренно искажённые метки.
DEV = (
    Example("dev-0", 0, 0), Example("dev-1", 1, 0),
    Example("dev-2", 2, 1), Example("dev-3", 3, 0),
    Example("dev-4", 4, 0), Example("dev-5", 5, 0),
    Example("dev-6", 6, 1), Example("dev-7", 7, 0),
    Example("dev-8", 8, 1), Example("dev-9", 9, 1),
)

def baseline(uid, x):
    return 0

def threshold_rule(uid, x):
    return int(x >= 6)

def build_memorizer(dev):
    labels = {row.uid: row.y for row in dev}
    return lambda uid, x: labels.get(uid, 0)

# Каждая стратегия видит только uid и x; истинная y ей не передаётся.
CANDIDATES = (
    ("baseline: всегда 0", baseline),
    ("rule: x >= 6", threshold_rule),
    ("memorizer: запомнить dev", build_memorizer(DEV)),
)
print("SYNTHETIC: dev-примеров:", len(DEV), "кандидатов:", len(CANDIDATES))

SYNTHETIC: dev-примеров: 10 кандидатов: 3


### Протокол выбора фиксируем до сравнения

1. Проверяем ровно **три** кандидата на одних и тех же 10 примерах dev: бюджет — **30 предсказаний**. Подготовка запоминателя использует только dev.
2. Побеждает наибольшая accuracy на dev; при равенстве — кандидат, стоящий раньше в `CANDIDATES`. Правило заранее задано.
3. Фиксируем имя и функцию победителя. Функция выбора получает **только dev**, а test ещё не создан в памяти при запуске сверху вниз.
4. Test раскрывается после фиксации. По нему нельзя менять победителя, кандидатов, порог или правило сравнения. Повторная разработка потребовала бы нового отложенного набора.

Это наглядная граница протокола, **не защита от чтения ячеек человеком**. Для настоящего слепого тестирования данные нужно хранить вне доступного разработчику контекста.

In [9]:
def score(candidate, rows):
    predictions = tuple(candidate(row.uid, row.x) for row in rows)
    assert all(type(prediction) is int and prediction in (0, 1) for prediction in predictions)
    return sum(prediction == row.y for prediction, row in zip(predictions, rows)) / len(rows)

def select_on_dev(candidates, dev):
    results = tuple((name, score(candidate, dev)) for name, candidate in candidates)
    best_index = max(range(len(results)), key=lambda index: results[index][1])
    return candidates[best_index], results

FROZEN_CHOICE, DEV_RESULTS = select_on_dev(CANDIDATES, DEV)
selected_name, selected_function = FROZEN_CHOICE
for name, accuracy in DEV_RESULTS:
    print(f"{name:30s} dev={accuracy:.0%}")
print("Выбор зафиксирован ДО test:", selected_name)
print("Бюджет сравнения на dev:", len(CANDIDATES) * len(DEV), "предсказаний")
assert selected_name == "memorizer: запомнить dev"
assert tuple(value for _, value in DEV_RESULTS) == (0.6, 0.8, 1.0)

baseline: всегда 0             dev=60%
rule: x >= 6                   dev=80%
memorizer: запомнить dev       dev=100%
Выбор зафиксирован ДО test: memorizer: запомнить dev
Бюджет сравнения на dev: 30 предсказаний


### Однократное раскрытие фиксированного test

Победил запоминатель: **10/10 на dev**, хотя он игнорирует $x$. Будет ли это улучшением на новых примерах?

В следующей ячейке впервые определяется заранее заданный test. Сначала вычисляется итог только зафиксированного победителя. Затем мы показываем остальные строки **для учебного разбора**. Это дополнительная диагностика после выбора, а не новый этап отбора; `FROZEN_CHOICE` остаётся прежним. Бюджет test: 10 предсказаний победителя + 20 для двух остальных кандидатов.

После просмотра этой таблицы test уже известен: нельзя править стратегии и продолжать называть повторное сравнение независимым. Для новой итерации нужен новый скрытый набор.

In [10]:
# Заранее сконструированный test; он не вычисляется из результатов выбора.
TEST = (
    Example("test-0", 0.25, 0), Example("test-1", 1.25, 0),
    Example("test-2", 2.25, 0), Example("test-3", 3.25, 0),
    Example("test-4", 4.25, 0), Example("test-5", 5.25, 0),
    Example("test-6", 6.25, 1), Example("test-7", 7.25, 1),
    Example("test-8", 8.25, 1), Example("test-9", 9.25, 1),
)
assert {row.uid for row in DEV}.isdisjoint(row.uid for row in TEST)
selected_test_score = score(selected_function, TEST)
print(f"Итог фиксированного победителя: {selected_name}, test={selected_test_score:.0%}")

test_results = {selected_name: selected_test_score}
for name, candidate in CANDIDATES:
    if name != selected_name:
        test_results[name] = score(candidate, TEST)
print("\nSYNTHETIC / диагностическая таблица после фиксации выбора")
print(f"{'Кандидат':30s} {'dev':>6} {'test':>6}")
for name, dev_score in DEV_RESULTS:
    print(f"{name:30s} {dev_score:6.0%} {test_results[name]:6.0%}")
assert FROZEN_CHOICE == (selected_name, selected_function)
assert selected_test_score == 0.6
assert test_results["rule: x >= 6"] == 1.0
print("Потеря запоминателя dev → test: 40 процентных пунктов.")

Итог фиксированного победителя: memorizer: запомнить dev, test=60%

SYNTHETIC / диагностическая таблица после фиксации выбора
Кандидат                          dev   test
baseline: всегда 0                60%    60%
rule: x >= 6                      80%   100%
memorizer: запомнить dev         100%    60%
Потеря запоминателя dev → test: 40 процентных пунктов.


### Что показал эксперимент и чего он не показал

На этих **фиксированных синтетических данных** запоминатель поднял dev с 60% до 100%, но на test сохранил 60%. Обобщающее правило получило 80% на шумном dev и 100% на test. Конструкция специально делает подгонку выгодной для отбора; это демонстрация механизма, а не оценка вероятности переобучения в реальном проекте.

**Самопроверка перед раскрытием ответов:**

1. Можно ли теперь назвать `rule` победителем нашего исходного dev-протокола?
2. Почему 100% на dev не доказывает улучшения на новых задачах?
3. Что менять в следующем исследовании: кандидатов, данные, протокол — и где взять свидетельства?

<details><summary>Ответы для обсуждения</summary>

1. Нет. Исходный протокол выбрал запоминатель. Если выбрать `rule` после просмотра test, это будет уже выбор с использованием test. Его качество потребует новой независимой проверки.
2. Запоминатель использовал идентификаторы и метки знакомых примеров, включая шум. На новых идентификаторах он возвращает 0. Высокая dev-метрика описывает совпадение с dev, а не универсальную способность решать задачу.
3. Можно проверить качество разметки и ввести валидацию на нескольких разбиениях внутри данных разработки. Решения принимаются на dev; новый итоговый test сохраняется скрытым. Из этого маленького примера нельзя получить универсальную гарантию.

</details>

## Итоги

Объясните по одной строке трассы: **какая информация была доступна → кто предложил действие → кто его проверил и исполнил → почему работа остановилась**. Затем найдите в синтетическом эксперименте момент фиксации победителя.

Примеры выполняются без LLM и внешних сервисов. Для работы с реальными сервисами также потребуются обработка исключений инструментов, таймауты, защита секретов и более содержательная оценка результата.

Чтобы выполнить всё заново, выберите **Restart Kernel → Run All**. Данные фиксированы, случайности нет. Повторный просмотр известного test не является новым независимым исследованием. Дополнительный вопрос: если `source` уже есть в истории, а `room` нет, какое действие вернёт наша политика? Ответ: `lookup(room)`.
